# V-JEPA 2 on MMAD — frozen visual representation benchmark

V-JEPA 2 is a visual encoder, not a language decoder, so reporting a zero-shot MMAD multiple-choice score would be invalid. This notebook instead measures what its frozen representation actually supports: anomaly detection, object-category recognition, and anomaly generalization to held-out object categories. Each static image is repeated into the model's 64-frame input.


In [ ]:
%pip install -q -U "transformers>=5.0.0" accelerate scikit-learn pandas matplotlib seaborn remotezip requests pillow


In [ ]:
import os,sys,json,time,hashlib,shutil,pickle,subprocess
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from transformers import AutoModel,AutoVideoProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler,normalize
from sklearn.metrics import accuracy_score,f1_score,balanced_accuracy_score,confusion_matrix,average_precision_score,roc_auc_score,brier_score_loss
from sklearn.model_selection import train_test_split

WORK=Path(os.environ.get('VJEPA_WORKDIR','/kaggle/working/vjepa_mmad' if Path('/kaggle/working').exists() else './vjepa_mmad')).resolve(); WORK.mkdir(parents=True,exist_ok=True)
REPO=WORK/'mini-world-model'; DATA=WORK/'data'; CACHE=WORK/'archive_cache'; OUT=WORK/'outputs'
for p in (DATA,CACHE,OUT):p.mkdir(parents=True,exist_ok=True)
url='https://github.com/anhsown/mini-world-model.git'
if REPO.exists():subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
else:subprocess.run(['git','clone','--depth','1',url,str(REPO)],check=True)
BASE=REPO/'research/mmad_model_benchmark';sys.path.insert(0,str(BASE));print('workdir',WORK)


In [ ]:
# Canonical full MMAD manifest and 8,366 neutral-named unique images.
subprocess.run([sys.executable,str(BASE/'prepare_full.py'),'--output',str(DATA),'--cache',str(CACHE),'--range-download'],cwd=BASE,check=True)
manifest=json.loads((DATA/'full_manifest.json').read_text(encoding='utf-8')); assert len(manifest['records'])==39670 and manifest['unique_images']==8366
by_image={}
for r in manifest['records']:
    by_image.setdefault(r['image_file'],{'image_file':r['image_file'],'source_dataset':r['source_dataset'],'category':r['category'],'is_normal':bool(r['is_normal'])})
df=pd.DataFrame(by_image.values()).sort_values('image_file').reset_index(drop=True); df['neutral_id']=[f'image_{i+1:06d}' for i in range(len(df))]
assert len(df)==8366 and all((DATA/p).exists() for p in df.image_file)
df['sha256']=[hashlib.sha256((DATA/p).read_bytes()).hexdigest() for p in df.image_file]
print({'images':len(df),'questions':len(manifest['records']),'exact_duplicate_images':int(df.sha256.duplicated().sum())});display(df.drop(columns=['image_file','sha256']).head())


In [ ]:
MODEL_ID='facebook/vjepa2-vitl-fpc64-256';assert torch.cuda.is_available(),'GPU required'
device=torch.device('cuda');processor=AutoVideoProcessor.from_pretrained(MODEL_ID);model=AutoModel.from_pretrained(MODEL_ID,torch_dtype=torch.float16).to(device).eval()
print(MODEL_ID,'parameters_M',round(sum(p.numel() for p in model.parameters())/1e6,1))
def embed_image(path):
    rgb=np.asarray(Image.open(path).convert('RGB')); frames=np.repeat(rgb[None,...],64,axis=0); video=torch.from_numpy(frames.copy()).permute(0,3,1,2)
    inputs=processor(video,return_tensors='pt')['pixel_values_videos'].to(device,dtype=torch.float16)
    with torch.inference_mode(): features=model.get_vision_features(inputs)
    return features.mean(dim=1).float().cpu().numpy()[0]
cache=WORK/'mmad_vjepa_embeddings.pkl';saved=pickle.loads(cache.read_bytes()) if cache.exists() else {}
limit=int(os.environ.get('MMAD_IMAGE_LIMIT','0')); active=df.iloc[:limit or None]
for i,row in active.iterrows():
    if row.neutral_id not in saved:saved[row.neutral_id]=embed_image(DATA/row.image_file)
    if (i+1)%50==0 or i+1==len(active):
        tmp=cache.with_suffix('.tmp');tmp.write_bytes(pickle.dumps(saved));tmp.replace(cache);print(i+1,'/',len(active),flush=True)
assert len(active)==len(df),'Set MMAD_IMAGE_LIMIT=0 for the reported full benchmark.'
X=np.stack([saved[k] for k in df.neutral_id]);print(X.shape)


In [ ]:
def tune_probe(X,y,tr,va,te):
    best=None
    for C in (0.01,0.1,1,10):
        clf=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=3000,class_weight='balanced'))
        clf.fit(X[tr],y[tr]);score=balanced_accuracy_score(y[va],clf.predict(X[va]))
        if best is None or score>best[0]:best=(score,C)
    C=best[1];fit=tr|va;clf=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=3000,class_weight='balanced'));clf.fit(X[fit],y[fit]);pred=clf.predict(X[te]);proba=clf.predict_proba(X[te])
    return {'C':C,'accuracy':accuracy_score(y[te],pred),'balanced_accuracy':balanced_accuracy_score(y[te],pred),'macro_f1':f1_score(y[te],pred,average='macro'),'truth':y[te],'pred':pred,'proba':proba}
idx=np.arange(len(df)); y_anom=(~df.is_normal).astype(int).values
tr,temp=train_test_split(idx,test_size=.30,random_state=20260729,stratify=y_anom);va,te=train_test_split(temp,test_size=.50,random_state=20260729,stratify=y_anom[temp])
masks=lambda a:np.isin(idx,a)
id_result=tune_probe(X,y_anom,masks(tr),masks(va),masks(te))
# Strong OOD protocol: no object category is shared across train, validation and test.
cats=np.array(sorted(df.category.unique()));rng=np.random.default_rng(20260729);rng.shuffle(cats);a=int(.7*len(cats));b=int(.85*len(cats));tc,vc,xc=set(cats[:a]),set(cats[a:b]),set(cats[b:])
tm=df.category.isin(tc).values;vm=df.category.isin(vc).values;xm=df.category.isin(xc).values
assert not(tc&vc or tc&xc or vc&xc)
ood_result=tune_probe(X,y_anom,tm,vm,xm)
# Object recognition on categories with enough examples, split at image level.
counts=df.category.value_counts();keep=df.category.isin(counts[counts>=20].index).values;ki=idx[keep];y_cat=df.category.astype('category').cat.codes.values
tr2,temp2=train_test_split(ki,test_size=.30,random_state=20260729,stratify=y_cat[ki]);va2,te2=train_test_split(temp2,test_size=.50,random_state=20260729,stratify=y_cat[temp2])
cat_result=tune_probe(X,y_cat,masks(tr2),masks(va2),masks(te2))
def binary_metrics(result):
    y=result['truth'];p=result['proba'][:,1];conf=np.maximum(p,1-p);err=(result['pred']!=y).astype(float);order=np.argsort(-conf);aurc=float(np.mean(np.cumsum(err[order])/np.arange(1,len(err)+1)))
    ece=0.0
    for lo in np.linspace(0,1,10,endpoint=False):
        m=(conf>=lo)&(conf<lo+.1)
        if m.any():ece+=m.mean()*abs((result['pred'][m]==y[m]).mean()-conf[m].mean())
    pred=result['pred'];return {'auprc':average_precision_score(y,p),'auroc':roc_auc_score(y,p),'brier':brier_score_loss(y,p),'ece':float(ece),'aurc':aurc,'miss_rate':float(((pred==0)&(y==1)).sum()/max((y==1).sum(),1)),'false_positive_rate':float(((pred==1)&(y==0)).sum()/max((y==0).sum(),1))}
summary={'model':MODEL_ID,'protocol':'frozen_encoder_linear_probe','capability_mapping':{'B1':'object/state recognition; primary=macro_f1','B3':'static anomaly proxy; primary=auprc','B10':'partial held-out-category calibration/risk-coverage proxy only'},'images':len(df),'exact_duplicate_images':int(df.sha256.duplicated().sum()),'anomaly_random_image_split':{k:id_result[k] for k in ('C','accuracy','balanced_accuracy','macro_f1')},'anomaly_heldout_category':{k:ood_result[k] for k in ('C','accuracy','balanced_accuracy','macro_f1')},'object_category_random_image_split':{k:cat_result[k] for k in ('C','accuracy','balanced_accuracy','macro_f1')},'heldout_categories':{'train':len(tc),'val':len(vc),'test':len(xc)}}
summary['anomaly_random_image_split'].update(binary_metrics(id_result));summary['anomaly_heldout_category'].update(binary_metrics(ood_result))
print(json.dumps(summary,indent=2))


In [ ]:
# Similarity audit and visual report.
A=normalize(X[tm]);B=normalize(X[xm]);max_sim=(B@A.T).max(axis=1);summary['ood_test_to_train_cosine']={'mean_max':float(max_sim.mean()),'p95_max':float(np.quantile(max_sim,.95)),'above_0_999':int((max_sim>.999).sum())}
(OUT/'vjepa2_mmad_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
import matplotlib.pyplot as plt,seaborn as sns
fig,axes=plt.subplots(1,3,figsize=(18,5));names=['Anomaly ID','Anomaly OOD category','Object category'];vals=[id_result['balanced_accuracy']*100,ood_result['balanced_accuracy']*100,cat_result['balanced_accuracy']*100]
axes[0].bar(names,vals,color=['#76b900','#e15759','#4c78a8']);axes[0].set_ylabel('Balanced accuracy (%)');axes[0].tick_params(axis='x',rotation=20);axes[0].set_title('Frozen V-JEPA probe results')
sns.heatmap(confusion_matrix(ood_result['truth'],ood_result['pred']),annot=True,fmt='d',cmap='Blues',ax=axes[1]);axes[1].set_title('OOD anomaly confusion')
axes[2].hist(max_sim,bins=30,color='#f28e2b');axes[2].set_title('OOD max test-to-train cosine');axes[2].set_xlabel('cosine similarity')
plt.tight_layout();plt.savefig(OUT/'vjepa2_mmad_analysis.png',dpi=160,bbox_inches='tight');plt.show()
release=WORK/'release';release.mkdir(exist_ok=True);shutil.copy2(OUT/'vjepa2_mmad_metrics.json',release/'vjepa2_mmad_metrics.json');shutil.copy2(OUT/'vjepa2_mmad_analysis.png',release/'vjepa2_mmad_analysis.png')
archive=shutil.make_archive(str(WORK.parent/'vjepa2_mmad_artifacts'),'zip',release);print('RESULT',archive)
